This is the high-level flow of the data ETL pipeline
- Read data from API(SQL database)
- Write to azure storage container
- Convert to parquet file
- Translate non-English text to English
- Create data subsets for Azure Cognitive Search upload

Import necessary libraries

In [ ]:
from gettext import translation
from requests.exceptions import HTTPError
from azure.ai.translation.text import TextTranslationClient
from azure.ai.translation.text.models import InputTextItem
from azure.core.exceptions import HttpResponseError, ResourceNotFoundError, AzureError
import os
import argparse
import json
import requests
import re
import warnings
import time
import pytz
from pytz import timezone
import pandas as pd
from io import BytesIO,StringIO
from dotenv import dotenv_values, load_dotenv
from msal import ConfidentialClientApplication
from azure.identity import EnvironmentCredential, ManagedIdentityCredential
from azure.keyvault.secrets import SecretClient
from azure.ai.ml.entities import AzureBlobDatastore, AccountKeyConfiguration
from azure.storage.blob import BlobServiceClient
from azure.ai.ml import MLClient
from datetime import datetime
from langdetect import detect, LangDetectException
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning
warnings.filterwarnings('ignore', category=MarkupResemblesLocatorWarning)

Define the global variables that will be used throughout this file

In [ ]:
credential = None
secret_client = None
date_data = None
folder_name = None
df_measures_orig = None
rest_api_client_secret_key_name = "<secret_key_name>"
storage_account_secret_key_name = "<storage_account_secret_key_name>"
region = "<region>"
resource_id = "/subscriptions/<subscription_id>/resourceGroups/<resource_group>/providers/Microsoft.CognitiveServices/accounts/<resource_name>"
endpoint = "https://<resource_name>.cognitiveservices.azure.com/"
data_frames = {}

In [ ]:
def initialize(subscription_id, resource_group, workspace_name, env_config_file, client_id):
    """
    Initializes the Azure Machine Learning workspace and client.

    Args:
        subscription_id (str): The Azure subscription ID.
        resource_group (str): The name of the resource group containing the workspace.
        workspace_name (str): The name of the Azure Machine Learning workspace.
        env_config_file (str): The path to the environment configuration file. If 'None', the variables passed in will be used.

    Returns:
        None
    """

    global secret_client, credential

    try:
        default_client_id = os.environ.get("DEFAULT_IDENTITY_CLIENT_ID")
        if default_client_id is not None:
            os.environ["AZURE_CLIENT_ID"] = default_client_id

        credential = ManagedIdentityCredential(client_id=client_id)

        if env_config_file != 'None':
            if os.path.isfile(env_config_file):
                config = dotenv_values(env_config_file)
                os.environ["AZURE_TENANT_ID"] = config["AZURE_TENANT_ID"]
                os.environ["AZURE_CLIENT_ID"] = config["AZURE_CLIENT_ID"]
                os.environ["AZURE_CLIENT_SECRET"] = config["AZURE_CLIENT_SECRET"]
                credential = EnvironmentCredential()

        print('Connected to Azure ML workspace')

        ml_client = MLClient(credential, subscription_id,
                             resource_group, workspace_name)
        workspace = ml_client.workspaces.get()
        key_vault_name = workspace.key_vault.split('/')[-1]
        key_vault_url = f"https://{key_vault_name}.vault.azure.net/"
        secret_client = SecretClient(
            vault_url=key_vault_url, credential=credential)

        print('Initialization done')
    except Exception as e:
        print(f"Initialization failed: {e}")
        return None

Get access token to authenticate with the web application which is the source of our Data

In [ ]:
def get_access_token(rest_api_tenant_id, rest_api_client_id, max_retries=3, backoff_factor=2):
    """ 
    This function is used to perform OAuth authorization and obtain the access token.

    Args:
        rest_api_tenant_id (str): Tenant ID for OAuth.
        rest_api_client_id (str): Client ID for OAuth.
        max_retries (int): Maximum number of retries for acquiring the token.
        backoff_factor (int): Factor for exponential backoff (e.g., 2 for doubling the wait time).

    Returns:
        string : Bearer token for the API request.
    """

    # Using key vault to obtain client secret
    client_id = rest_api_client_id
    tenant_id = rest_api_tenant_id
    client_credential = secret_client.get_secret(
        rest_api_client_secret_key_name).value

    authority = "https://login.microsoftonline.com/{tenant_id}/".format(
        tenant_id=tenant_id)
    scope = ["<scope_url>/.default"]
    grant_type = "client_credentials"

    # Initialize the MSAL client
    app = ConfidentialClientApplication(
        client_id=client_id,
        client_credential=client_credential,
        authority=authority
    )
    retries = 0
    while retries < max_retries:
        try:
            accounts = app.get_accounts()

            if accounts:
                result = app.acquire_token_silent(
                    scope=scope, account=accounts[0])
            else:
                result = app.acquire_token_for_client(
                    scopes=scope,
                )

            # Check if the authentication was successful
            if "access_token" in result:
                return result["access_token"]

            error = result.get("error")
            error_description = result.get("error_description")
            print(f"Token acquisition failed: {error} - {error_description}")

        except HTTPError as e:
            # Catch HTTP-related errors, which could indicate transient issues
            print(f"HTTP error occurred: {e}")
            return

        # Increment retry count and apply exponential backoff
        retries += 1
        wait_time = backoff_factor ** retries
        print(f"Retrying in {wait_time} seconds...")
        time.sleep(wait_time)

    # Raise an exception or return None if max retries are exceeded
    print("Authentication failed after maximum retries.")
    return None

Helper functions for naming of files in landing container

In [ ]:
def parse_date_pairs_from_blob(container_client, initial_dates_blob_name, file):
    """
    Parse start_date and end_date pairs from initial_dates.txt in Azure Blob Storage.

    Args:
        container_client (azure.storage.blob.ContainerClient): The container client to access the blob storage.
        initial_dates_blob_name (str): The name of the blob containing initial dates.
        file (str): The file name to be used in the API URL.

    Returns:
        list: A list of API URLs constructed from the date pairs.
    """

    try:
        # Delete initial_dates.txt after first load or add a date in the date.txt file which is currently 'None'.
        blob_client = container_client.get_blob_client(initial_dates_blob_name)
        file_content = blob_client.download_blob().readall().decode("utf-8")

        # Extract start_date and end_date pairs
        date_pairs = []
        API_URLs = []
        lines = file_content.split("\n")
        start_date, end_date = None, None

        for line in lines:
            if line.startswith("start_date="):
                start_date = line.split("=", 1)[1].strip()
            elif line.startswith("end_date="):
                end_date = line.split("=", 1)[1].strip()
                if start_date and end_date:
                    date_pairs.append((start_date, end_date))
                    start_date, end_date = None, None

        for start_date, end_date in date_pairs:
            api_url = "https://<database_api_url>/api/DataExport/{file}/From/{start_date}/To/{end_date}".format(
                file=file, start_date=start_date, end_date=end_date)
            API_URLs.append(api_url)

    except Exception as e:
        raise RuntimeError(
            f"Error reading {initial_dates_blob_name} from blob storage: {e}")

    return API_URLs

Read the data from the source (here it is an API endpoint of a SQL database)

In [ ]:
def read_data_from_api(rest_api_tenant_id, rest_api_client_id, max_retries=3, backoff_factor=3):
    """
    This function is for reading Users,Measures and User Actions data from API endpoint. 
    The arguments are passed to the get_Access_token() function to receive a token after authorization.

    Args:
        rest_api_tenant_id (str): Tenant ID of the Azure subscription
        rest_api_client_id (str): Client ID of the Service Principal
        max_retries (int): Maximum number of retries for the API request.
        backoff_factor (int): Factor for exponential backoff (e.g., 2 for doubling the wait time).

    The data is read in chunks/blocks as it is a huge data file and 
    reading in blocks helps not to run into "Memory" issue.

    Returns:
        str: Success message if the file is uploaded successfully, or error message if it fails.
    """

    global date_data
    Files = ['Users', 'Measures', 'UserActions']

    container_name = "landing"
    date_container_name = "dataeng-pipeline-date"
    date_blob_name = "date.txt"
    initial_dates_blob_name = "initial_dates.txt"
    utc = pytz.timezone('UTC')

    # Initialize the blob client
    blob_service_client = BlobServiceClient(
        account_url=f"https://<storage_account_name>.blob.core.windows.net", credential=credential)
    container_client = blob_service_client.get_container_client(
        date_container_name)
    # Check if the container exists, if not, create it --> date container
    if not container_client.exists():
        blob_service_client.create_container(date_container_name)
        print(f"Created container: {date_container_name}")

    # Check if the container exists, if not, create it --> landing container
    landing_container_client = blob_service_client.get_container_client(
        container_name)
    if not landing_container_client.exists():
        blob_service_client.create_container(container_name)
        print(f"Created container: {container_name}")

    # Get date value from text file in blob container
    blob_client = container_client.get_blob_client(date_blob_name)
    try:
        blob_content = blob_client.download_blob().readall().decode("utf-8")

        lines = blob_content.splitlines()
        for line in lines:
            if line.startswith('pipeline_run_date_with_time='):
                date_data = line.split('=', 1)[1]  # Return value after the key

    except Exception as e:
        print(f"Error reading blob: {e}")
        return None

    for file in Files:
        if file != "UserActions":
            blob_name = f"{file}/{file}_from_API.json"
            api_url = "https://<database_api_url>/api/DataExport/{file}".format(
                file=file)
            api_url_list = [api_url]
        else:
            blob_name_template = "UserActions/{Year}/Actions_{FromDate}_To_{ToDate}.json"
            start_date = date_data
            # Last end date value should be stored as a start date in the date_txt --> check if needed
            if start_date == 'None':
                api_url_list = parse_date_pairs_from_blob(
                    container_client, initial_dates_blob_name, file)  # add the blob_name as dynamic
            else:
                date_object = datetime.strptime(date_data, "%Y-%m-%d %H:%M:%S")
                date_part = date_object.date()
                date_string = date_part.strftime("%Y-%m-%d")
                start_date = date_string

                end_date = format(datetime.now(), "%Y-%m-%d")

                end_date_with_time = datetime.now(utc).strftime("%Y-%m-%d %H:%M:%S")
                date_key_value_pair = f"pipeline_run_date_with_time={end_date_with_time}"

                year = datetime.strptime(start_date, "%Y-%m-%d").year
                # Replace '-' in dates with '_' to make the filename safe
                blob_name = blob_name_template.format(Year=year,
                                                      FromDate=start_date.replace(
                                                          '-', '_'),
                                                      ToDate=end_date.replace('-', '_'))

                api_url = "https://<database_api_url>/api/DataExport/{file}/From/{start_date}/To/{end_date}".format(
                    file=file, start_date=start_date, end_date=end_date)
                api_url_list = [api_url]

        token = get_access_token(rest_api_tenant_id, rest_api_client_id)
        headers = {
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json'
        }

        for api_url_item in api_url_list:
            if file == "UserActions":
                # Dynamically extract start_date and end_date from the API URL for naming
                parsed_start_date = api_url_item.split(
                    "/From/")[1].split("/To/")[0]
                parsed_end_date = api_url_item.split("/To/")[1]

                year = datetime.strptime(parsed_start_date, "%Y-%m-%d").year
                blob_name = blob_name_template.format(
                    Year=year,
                    FromDate=parsed_start_date.replace("-", "_"),
                    ToDate=parsed_end_date.replace("-", "_"))

            retries = 0
            while retries < max_retries:
                try:
                    response = requests.get(
                        api_url_item, headers=headers, stream=True)
                    if response.status_code == 200:
                        # Save the data from the API response to a blob
                        blob_client = blob_service_client.get_blob_client(
                            container=container_name, blob=blob_name)

                        chunk_size = 4 * 1024 * 1024  # 4MB chunks
                        block_list = []
                        block_id_prefix = "block"

                        for i, chunk in enumerate(response.iter_content(chunk_size=chunk_size)):
                            if chunk:  # Only upload non-empty chunks
                                block_id = f'{block_id_prefix}{i:07d}'.encode(
                                    'utf-8')  # Create block ID
                                # Stage each chunk as a block
                                blob_client.stage_block(block_id, chunk)
                                block_list.append(block_id)

                        blob_client.commit_block_list(block_list)
                        print(
                            f"File uploaded to {blob_name} in container {container_name} successfully.")

                        if file == "UserActions":
                            if start_date != 'None':
                                # Update the date value in the blob storage only if the GET request for UserActions is successful
                                container_client.upload_blob(
                                    name=date_blob_name, data=date_key_value_pair, overwrite=True)
                                print(
                                    f"Date {date_key_value_pair} uploaded to {date_container_name}.")
                        break

                    elif response.status_code in {500, 502, 503, 504}:
                        print(
                            f"Server error (status code: {response.status_code}). Retrying...")

                    else:
                        print(
                            f"Failed to retrieve file. Status code: {response.status_code}")
                        return None

                except requests.exceptions.RequestException as e:
                    print(f"Network error occurred: {e}. Retrying...")

                retries += 1
                wait_time = backoff_factor ** retries
                print(f"Retrying in {wait_time} seconds...")
                time.sleep(wait_time)

            if retries == max_retries:
                print("Failed to fetch data from the API after maximum retries.")
                return None

Helper function to keep consistency in column naming

In [ ]:
def to_camel_case(col):
    """
    This function is to convert Pascal Case to camel Case. 

    Args:
        col (str): COlumn Name in the dataframe

    Returns:
        str: Column name converted to camelCase.
    """
    col = re.sub(r"([A-Z])", r" \1", col).split()  # Split at capital letters
    return col[0].lower() + ''.join(word.capitalize() for word in col[1:])

Archiving .json files after conversion to .parquet to reduce the storage space

In [ ]:
def move_user_actions_to_archive(blob_service_client, source_container_name, archive_container_name):
    """
    Moves the UserActions files from the landing container to the archive container.

    This function is called after the translation of the UserActions files is complete. It iterates through the 
    UserActions files in the landing container, moves them to the archive container, and deletes the original files 
    from the landing container to maintain a clean state.

    Args:
        blob_service_client (BlobServiceClient): The client for the blob service.
        source_container_name (str): The name of the container that stores the UserActions files.
        target_container_name (str): The name of the container where the UserActions files will be moved.

    Returns:
        None
    """

    source_container_client = blob_service_client.get_container_client(
        source_container_name)
    archive_container_client = blob_service_client.get_container_client(
        archive_container_name)

    # Check if the archive container exists, if not, create it
    if not archive_container_client.exists():
        blob_service_client.create_container(archive_container_name)
        print(f"Created archive container: {archive_container_name}")

    user_actions_blobs = [blob.name for blob in source_container_client.list_blobs(
        name_starts_with="UserActions/") if blob.name.endswith('.json')]

    if not user_actions_blobs:
        print("No UserActions files found to move.")
        return None

    for blob_name in user_actions_blobs:
        try:
            blob_client = source_container_client.get_blob_client(blob_name)
            blob_data = blob_client.download_blob().readall()
            archive_blob_client = archive_container_client.get_blob_client(
                blob_name)
            archive_blob_client.upload_blob(blob_data, overwrite=True)
            blob_client.delete_blob()
            print(f"Moved blob {blob_name} to {archive_container_name}.")
        except Exception as e:
            print(f"Error moving blob {blob_name}: {e}")
            return None

    print("All UserActions files have been moved to the archive container.")

Convert the .json to .parquet files which reduces storage space and is optimized for data retrieving

In [ ]:
def convert_to_parquet(source_container_name, target_container_name, archive_container_name):
    """
    This function is for converting Users,Measures JSON files to parquet files. For User Actions files, the nested JSON is split into different Actions parquet files.
    Conversion to parquet files helps in compressing the JSON files and also quick retrieval of data for further processing.

    Args:
        source_container_name (str): Container consisting of JSON files
        target_container_name (str): Container to save parquet files

    Returns:
        None
    """

    global folder_name

    folder_name = 'new_data'
    # Initialize the blob client
    blob_service_client = BlobServiceClient(
        account_url=f"https://<storage_account_name>.blob.core.windows.net", credential=credential)
    source_container_client = blob_service_client.get_container_client(
        source_container_name)
    if not source_container_client.exists():
        blob_service_client.create_container(source_container_name)
        print(f"Created container: {source_container_name}")

    target_container_client = blob_service_client.get_container_client(
        target_container_name)
    if not target_container_client.exists():
        blob_service_client.create_container(target_container_name)
        print(f"Created container: {target_container_name}")

    json_blobs = list(source_container_client.list_blobs())
    
    if not json_blobs:
        print("No JSON blobs found in the source container. Exiting function.")
        return None

    # Users and Measures JSON files
    for blob in json_blobs:
        if blob.name.endswith('.json'):
            top_folder_name = blob.name.split('/')[0]

            if top_folder_name in ['Users', 'Measures']:
                # Check if the folder is either "Users" or "Measures", then simple conversion to parquet format
                print(f"Processing {blob.name}...")

                try:
                    json_blob_client = source_container_client.get_blob_client(
                        blob.name)
                    json_data = json_blob_client.download_blob().readall().decode("utf-8")

                    df = pd.read_json(StringIO(json_data))
                    df.columns = [to_camel_case(col) for col in df.columns]

                    # 🔹 Enforce string logic for transferable
                    if top_folder_name == 'Measures' and "transferable" in df.columns:
                        df["transferable"] = (
                            df["transferable"]
                            .astype(str)
                            .str.strip()
                            .str.lower()
                            .replace({
                                "1": "Yes",
                                "2": "No",
                                "0": "No",
                                "yes": "Yes",
                                "no": "No",
                                "none": pd.NA,
                                "null": pd.NA,
                                "nan": pd.NA,
                                "": pd.NA
                            })
                            .astype("string")
                        )

                    parquet_buffer = BytesIO()
                    df.to_parquet(parquet_buffer, index=False)
                    parquet_buffer.seek(0)

                    parquet_blob_name = f"{folder_name}/{blob.name.replace('.json', '.parquet')}"
                    target_blob_client = target_container_client.get_blob_client(
                        parquet_blob_name)
                    target_blob_client.upload_blob(
                        parquet_buffer.getvalue(), overwrite=True)

                    print(
                        f"Uploaded {parquet_blob_name} to {target_container_name}.")
                except FileNotFoundError as e:
                    print(f"Error downloading JSON blob {blob.name}: {e}")
                    return None
                except ValueError as e:
                    print(f"Error parsing JSON data from {blob.name}: {e}")
                    return None
                except Exception as e:
                    print(f"Unexpected error processing {blob.name}: {e}")
                    return None

        # For UserActions files
            # Replace ''UserActions/2024/Actions_2024_10_30_To_2024_10_31.json' with '/'
            elif top_folder_name == 'UserActions' and '/' in blob.name:
                parts = blob.name.split('/')
                # Extract parent and child folder names
                # The first part is the parent folder
                sub_folder_name = parts[1]

                print(f"Processing {blob.name}...")
                try:
                    json_blob_client = source_container_client.get_blob_client(
                        blob.name)
                    json_data = json_blob_client.download_blob().readall().decode("utf-8")

                    json_data_dict = json.loads(json_data)

                    # Process records and convert to Parquet
                    for key, records in json_data_dict.items():
                        if records:  # Only process if the list is not empty
                            df = pd.DataFrame(records)
                            df.columns = [to_camel_case(
                                col) for col in df.columns]
                            parquet_blob_name = f"{folder_name}/{top_folder_name}/{sub_folder_name}/{key}/{blob.name.split('/')[-1].rstrip('.json')}_{key}.parquet"

                            # Convert the DataFrame to a Parquet file in memory
                            parquet_buffer = BytesIO()
                            df.to_parquet(parquet_buffer, index=False)
                            parquet_buffer.seek(0)

                            # Upload the Parquet file to the target container
                            target_blob_client = target_container_client.get_blob_client(
                                parquet_blob_name)
                            target_blob_client.upload_blob(
                                parquet_buffer.getvalue(), overwrite=True)

                            print(
                                f"Uploaded {parquet_blob_name} to {target_container_name}.")
                except (json.JSONDecodeError, Exception) as e:
                    print(f"Error processing {blob.name}: {e}")
                    return None

    move_user_actions_to_archive(
        blob_service_client, source_container_name, archive_container_name)

    print("All JSON files have been processed and converted to Parquet.")

The below functions are to handle translation from any language to English . The English language utilize lesser tokens and the score is more precise.

1. Function to get divide the measures into equal amount of batches

In [ ]:
def get_valid_measures(last_processed_measure_id, batch_size=50):
    """
    This function divides the measures DataFrame into batches as per the batch size.
    It handles non-consecutive measure IDs with large gaps by processing the next available IDs.

    Args:
        last_processed_measure_id (int): The last processed ID for a single batch.
        batch_size (int): The batch of measures that should be processed in a single run.

    Returns:
        pd.DataFrame: Single batch of measures that need to be processed.
        int: Measure ID for the naming convention.
        int: The last processed ID for the batch.
    """

    # Save the initial last_processed_measure_id for naming convention
    measure_id = last_processed_measure_id
    unique_ids = sorted(df_measures_orig[df_measures_orig['measureId'] > last_processed_measure_id]['measureId'].unique())

    # If no more IDs to process, return an empty DataFrame
    if not unique_ids:
        return pd.DataFrame(), measure_id, last_processed_measure_id

    # Get the next batch of IDs
    current_batch_ids = unique_ids[:batch_size]
    df_measures = df_measures_orig[df_measures_orig['measureId'].isin(current_batch_ids)]

    # Update the last_processed_measure_id to the max ID in the current batch
    last_processed_measure_id = max(current_batch_ids)

    # If there are no more IDs after this batch, return an empty DataFrame next time
    if last_processed_measure_id == max(unique_ids):
        return df_measures, measure_id, last_processed_measure_id

    return df_measures, measure_id, last_processed_measure_id

2. To clean the text columns as they contain HTML tags and special characters, we can use the BeautifulSoup library to parse the HTML content and extract the text. Below is a function that takes a DataFrame and a list of columns to clean, and returns the cleaned DataFrame.

In [ ]:
def contains_html(text):
    """
    This function checks if the text has HTML tags. 

    Args:
        text(dataframe) : Dataframe containing the measures data.

    Returns:
        The search method returns a match object if it finds a match otherwise, it returns None. Wrapping this in bool() converts the result to True or False.
    """

    if pd.isna(text) or text.strip() == '':  # To ignore NULL and blank values
        return False
    htmlTagPattern = re.compile('<.*?>')
    return bool(htmlTagPattern.search(text))


def is_col_contain_html_tags(df_measures):
    """
    This function loops through the text columns in measures dataframe. 

    Args:
        df_measures(dataframe) : Dataframe containing the measures data.

    Returns:
        Cleaned dataframe without HTML tags.
    """

    cols = ['businessRequirement', 'targetState', 'initialState',
            'remarks', 'analysis', 'results', 'lessonsLearned']

    for col in cols:
        # Removes html tags even if one of the value is True
        if df_measures[col].apply(contains_html).any() == True:
            df_measures.loc[:, col] = df_measures[col].apply(remove_html_tags)

    return df_measures


def remove_html_tags(text):
    """This function removes HTML tags from the specific text columns. 

        Args:
            text(str) : The input text that may contain HTML tags.

        Returns:
            The cleaned text without HTML tags, or the original text if it’s NaN or blank.
    """
    if pd.isna(text) or text.strip() == '':  # To ignore NULL and blank values
        return text
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text()


def clean_text(text):
    """
    Cleans the input text by removing URLs and special characters.

    This function performs the following operations:
    1. Removes URLs that start with 'http', 'https', or 'www'.
    2. Removes special characters except for specific character sets:
       - CJK (Chinese, Japanese, Korean)
       - Latin
       - Vietnamese
       - Greek
       - Cyrillic
    3. Additionally, removes certain punctuation characters (., -, ...).

    Args:
        text (str): The input string to be cleaned.

    Returns:
        str: The cleaned text, with URLs and unwanted special characters removed.
    """

    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(
        r'[^\w\s\u4E00-\u9FFF\u3400-\u4DBF\u3040-\u309F\u30A0-\u30FF\uAC00-\uD7AF\u1100-\u11FF\uF900-\uFAFF\uFE30-\uFE4F\u00C0-\u00FF\u0100-\u017F\u0370-\u03FF\u0400-\u04FF]',
        '',
        text)
    text = re.sub(r'[.,-,...]', '', text)

    return text

3. Save the translated measures in a separate storage path

In [ ]:
def save_translated_measures(df_measures, measure_id, batch_size, translations_container_name):
    """
    Deletes the old blobs if exists.
    Uploads a DataFrame of translated measures to Azure Blob Storage as a Parquet file.

     Args:
         df_measures (pd.DataFrame): The DataFrame containing the translated measures to be uploaded.
         first_measure_id (int): The ID of the last processed measure, used to generate the file name.
         batch_size (int): The size of the batch, used to generate the file name.
         translations_container_name(str): Container to store the translated measures.

     Returns:
         None: This function does not return a value but uploads the DataFrame as a Parquet file to Blob Storage.

     Raises:
         Exception: Raises an exception if the blob upload fails or if there are issues with the connection.
    """

    blob_path = f'translated/Measures_translated_{measure_id+1}_{measure_id + batch_size}.parquet'

    try:
        # Initialize the Blob Service Client
        blob_service_client = BlobServiceClient(
            account_url="https://<storage_account_name>.blob.core.windows.net", credential=credential)
        
        translation_container_client = blob_service_client.get_container_client(
            translations_container_name)
        if not translation_container_client.exists():
            blob_service_client.create_container(translations_container_name)
            print(f"Created container: {translations_container_name}")

        blob_client = blob_service_client.get_blob_client(
            container=translations_container_name, blob=blob_path)

        # Check if the blob already exists and delete it if it does
        if blob_client.exists():
            blob_client.delete_blob()
            print(f"Deleted old blob: {blob_path}")

        # Convert DataFrame to Parquet
        parquet_file = BytesIO()
        df_measures.to_parquet(parquet_file, engine='pyarrow')
        parquet_file.seek(0)  # Reset stream position

        # Upload the blob
        blob_client.upload_blob(data=parquet_file, overwrite=True)
        print(f"Successfully uploaded {blob_path}.")

    except AzureError as e:
        print(f"Failed to upload the blob: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

4. Translate measures data using Azure Translator Service and save the translated data in the same parquet format in the blob storage.

In [ ]:
def translate_measures(target_container_name, translations_container_name):
    """
    This function is for translating non-english measures for the batch of measures received. 

    Args:
        target_container_name(str): Container that contains the measures parquet file to be translated.
        translations_container_name(str): Container to store the translated measures.

    Returns:
        None
    """

    global date_data, folder_name, df_measures_orig, region, resource_id, endpoint, pipeline_run_date_utc
    file_name = "Measures.parquet"
    # Global variables . Replace these values later with global variables
    # folder_name = 'new_data'
    # date_data = '2024-10-19 14:24:04'

    # date_only = date_data.split(' ')[0]
    pipeline_run_date = datetime.strptime(date_data, "%Y-%m-%d %H:%M:%S")  
    pipeline_run_date = pd.to_datetime(pipeline_run_date) # Does not change the timezone, instead to make the Timestamps timezone-aware
    pipeline_run_date_utc = pipeline_run_date.tz_localize('UTC')
    
    blob_service_client = BlobServiceClient(
        account_url=f"https://<storage_account_name>.blob.core.windows.net", credential=credential)
    
    # Read measuresIds from translated-measures container
    translations_container_client = blob_service_client.get_container_client(
        translations_container_name)
    translations_blob_client = translations_container_client.get_blob_client(blob=file_name)
    if translations_blob_client.exists():
        blob_data = translations_blob_client.download_blob()
        df_translations = pd.read_parquet(BytesIO(blob_data.readall()))

    # Initialize the blob client for the source container
    source_container_client = blob_service_client.get_container_client(
        target_container_name)
    file_path = f"{folder_name}/Measures/Measures_from_API.parquet"
    json_blobs = source_container_client.list_blobs(name_starts_with=file_path)

    measures_blob = [blob for blob in json_blobs if blob.name.endswith("Measures_from_API.parquet")]
    if not measures_blob:
        print("No Measures_from_API.parquet file found in the source container.")
        return None
    blob = measures_blob[0]

    blob_client = source_container_client.get_blob_client(blob.name)
    blob_data = blob_client.download_blob().readall()
    df_measures_orig = pd.read_parquet(BytesIO(blob_data))

    # Converting date columns to datetime datatype
    df_measures_orig['lastUpdatedAt'] = pd.to_datetime(
        df_measures_orig['lastUpdatedAt'], errors='coerce')
    df_measures_orig['createdAt'] = df_measures_orig['createdAt'].apply(
        lambda x: x[:19] + 'Z' if isinstance(x, str) else x)
    df_measures_orig['createdAt'] = pd.to_datetime(
        df_measures_orig['createdAt'])
    df_measures_orig['endDate'] = pd.to_datetime(
        df_measures_orig['endDate'], errors='coerce')
    df_measures_orig['startDate'] = pd.to_datetime(
        df_measures_orig['startDate'], errors='coerce')

    # TODO: Confirm data types by the developers
    df_measures_orig['projectTypes'] = df_measures_orig['projectTypes'].astype(
        str)
    df_measures_orig['status'] = df_measures_orig['status'].astype(str)
    df_measures_orig['hardnessGrade'] = df_measures_orig['hardnessGrade'].astype(
        str)
    df_measures_orig['priority'] = df_measures_orig['priority'].astype(str)
    df_measures_orig['simpleScore'] = df_measures_orig['simpleScore'].astype(
        float)

    # Get measures only where 'createdAt' and 'lastUpdatedAt' >= last_pipeline_run_date
    # Logic to also check for measures ids in original measures which are not present in translated_measures
    if pipeline_run_date_utc is not None and pipeline_run_date_utc != 'None':
        df_measures_orig = df_measures_orig[(df_measures_orig['createdAt'] >= pipeline_run_date_utc) | (
            df_measures_orig['lastUpdatedAt'] >= pipeline_run_date_utc) | 
            (~df_measures_orig['measureId'].isin(df_translations['measureId']))]
        # Check if df_measures_orig is empty, if so, break
        if df_measures_orig.empty:
            print("No measures to process. Exiting.")
            return None
        
        # Sort by 'measureId' in ascending order to get the smallest measureId
        df_measures_orig = df_measures_orig.sort_values(by='measureId', ascending=True)
        last_processed_measure_id = df_measures_orig['measureId'].iloc[0] - 1
    else:
        last_processed_measure_id = 0

    if df_measures_orig.empty:
        print("No measures to translate.")
        return None

    is_df_measures_available = True

    while is_df_measures_available:
        df_measures, measure_id, last_processed_measure_id = get_valid_measures(
            last_processed_measure_id)

        if df_measures.empty:
            is_df_measures_available = False
            print("All measures have been processed!")
            break
        df_measures = df_measures.sort_values(by='measureId')

        # Remove HTML tags
        df_measures = is_col_contain_html_tags(df_measures)

        # Initialize the text translator client
        text_translator = TextTranslationClient(
            region=region, resource_id=resource_id, endpoint=endpoint, credential=credential)

        # TODO: Replace the logging code with Azure ML logging. Save in default datastore.
        # Connect to your workspace

        # Configure logging to write to a file, with time and error information
        # logging.basicConfig(filename='tmp/translation_errors.log',  # Name of the log file
        #                     level=logging.ERROR,               # Log only errors
        #                     format='%(asctime)s - %(levelname)s - %(message)s')

        # ws = Workspace.from_config()

        # # Get the custom datastore
        # datastore = Datastore.get(ws, datastore_name='your_custom_datastore')

        # # Upload the log file to the custom datastore
        # datastore.upload_files(
        #     files=[log_file],
        #     target_path='logs/',
        #     overwrite=True
        # )

        source_cols = ['name', 'description', 'businessRequirement', 'targetState',
                       'initialState', 'remarks', 'analysis', 'results', 'lessonsLearned']

        for col in source_cols:
            measures_list = df_measures[col].tolist()
            translated_col = []

            for item in measures_list:
                # Initialize translated_name for each item
                translated_name = item

                # Check if item is relevant for translation
                if item is None or item.strip() == '' or item.isdigit() or item == '-':
                    translated_col.append(translated_name)
                    continue

                # If the item is relevant, proceed with language detection and translation
                MAX_TEXT_LENGTH = 50000
                lang_item = item[:MAX_TEXT_LENGTH]
                processed_text = clean_text(lang_item)
                try:
                    lang = detect(processed_text)
                except LangDetectException as e:
                    print(f"Error detecting language: '{lang_item}' Col '{col}. Exception: {e}")
                    lang = 'unknown'

                # If language detected is not English, attempt translation
                if lang != 'en' and lang != 'unknown':
                    input_text_elements = [InputTextItem(text=item)]
                    try:
                        response = text_translator.translate(
                            body=input_text_elements, to_language=["en"])
                        translation = response[0] if response else None

                        # If translation is successful, use translated text
                        if translation:
                            translated_name = translation.translations[0].text
                        # If translation fails , roll back to original text
                        else:
                            translated_name = item
                    except HttpResponseError as http_err:
                        print(
                            f"HTTP error occurred during translation: {http_err.status_code} - {http_err.message}")
                        # On error, fallback to original text
                        translated_name = item
                    except Exception as e:
                        print(f"Error during translation: {e}")
                        translated_name = item

                else:
                    # If language is English, keep the original text
                    translated_name = item

                # Append the result (translated or original) to the translated column list
                translated_col.append(translated_name)

            translated_col = pd.Series(translated_col, index=df_measures.index)
            df_measures.loc[:, col] = translated_col

        save_translated_measures(df_measures, measure_id, len(
            df_measures), translations_container_name)


Data subsets used for updation of Azure Cognitive search

In [ ]:
def copy_new_or_updated_translation_files(translations_container_client, translations_subset_blob_name, df, timeformat):
    """
    Copies new or updated translation files to a specified blob storage location.
    This function checks if a blob with the specified name exists in the translations container.
    If it exists, the blob is deleted. Then, a new Parquet file containing the provided DataFrame
    is created and uploaded to the blob storage with a timestamped name.
    Args:
        translations_container_client (azure.storage.blob.ContainerClient): 
            The Azure Blob Storage container client used to interact with the translations container.
        translations_subset_blob_name (str): 
            The name of the blob where the translation subset will be stored.
        df (pandas.DataFrame): 
            The DataFrame containing the data to be saved as a Parquet file.
        timeformat (str): 
            A string representing the timestamp format to be appended to the blob name.
    Returns:
        None
    Raises:
        azure.core.exceptions.ResourceNotFoundError: 
            If the specified container or blob does not exist.
        azure.core.exceptions.HttpResponseError: 
            If there is an error during blob deletion or upload.
    Notes:
        - The function uses the PyArrow engine to write the DataFrame to a Parquet file.
        - The blob name includes a timestamp to ensure uniqueness and traceability.
    """

    blob_name = f"{translations_subset_blob_name}/new_or_updated_measures_subset_{timeformat}.parquet"
    blob_client = translations_container_client.get_blob_client(blob_name)
    
    # If the folder exists, delete all blobs within the folder before uploading the new one
    blobs_in_folder = translations_container_client.list_blobs(name_starts_with=f"{translations_subset_blob_name}/new_or_updated_measures_subset_")
    for blob in blobs_in_folder:
        blob_client_to_delete = translations_container_client.get_blob_client(blob.name)
        blob_client_to_delete.delete_blob()

    # Write DataFrame to BytesIO stream and upload to blob storage
    with BytesIO() as output_stream:
        df.to_parquet(output_stream, index=False, engine='pyarrow')
        output_stream.seek(0)
        blob_client.upload_blob(output_stream, overwrite=True)

    print("Translated measures subset for new/updated measures is created.")


def save_deleted_measures(deleted_measures,translations_container_client,translations_subset_blob_name, timeformat):
    """
    Saves a subset of deleted measures to a Parquet file and uploads it to a specified Azure Blob Storage location.
    Args:
        deleted_measures (pandas.DataFrame): A DataFrame containing the deleted measures to be saved.
        translations_container_client (azure.storage.blob.ContainerClient): The Azure Blob Storage container client used to access the blob.
        translations_subset_blob_name (str): The name of the blob where the deleted measures subset will be stored.
        timeformat (str): A timestamp or formatted string to include in the filename for versioning.
    Returns:
        None
    Side Effects:
        - Creates a Parquet file in memory containing the deleted measures.
        - Uploads the Parquet file to the specified Azure Blob Storage location.
        - Prints a confirmation message upon successful upload.
    """

    blob_name = f"{translations_subset_blob_name}/deleted_measures_subset_{timeformat}.parquet"
    blob_client = translations_container_client.get_blob_client(blob_name)
    
    # If the folder exists, delete all blobs within the folder before uploading the new one
    blobs_in_folder = translations_container_client.list_blobs(name_starts_with=f"{translations_subset_blob_name}/deleted_measures_subset_")
    for blob in blobs_in_folder:
        blob_client_to_delete = translations_container_client.get_blob_client(blob.name)
        blob_client_to_delete.delete_blob()

    # Write the deleted measures DataFrame to a BytesIO stream and upload it to blob storage
    with BytesIO() as output_stream:
        deleted_measures.to_parquet(output_stream, index=False, engine='pyarrow')
        output_stream.seek(0)
        blob_client.upload_blob(output_stream, overwrite=True)

    print("Deleted measures subset created.")

Delete all batch translation files after merging

In [ ]:
def delete_batch_translation_files(blob_service_client, translations_container_name):
    """
    Deletes the translated files in batches after merging them.

    This function is called after the merging of translated files is complete. It iterates through the 
    translated files in the specified container and deletes each file to free up storage space and 
    maintain a clean state in the container.

    Args:
        blob_service_client (BlobServiceClient): The client for the blob service.
        translations_container_name (str): The name of the container that stores the translated measures.

    Returns:
        None
    """

    container_client = blob_service_client.get_container_client(
        translations_container_name)
    parquet_blobs = [blob.name for blob in container_client.list_blobs(
        name_starts_with="translated/") if blob.name.endswith('.parquet')]

    for blob_name in parquet_blobs:
        try:
            blob_client = container_client.get_blob_client(blob_name)
            blob_client.delete_blob()
            print(f"Deleted blob: {blob_name}")
        except Exception as e:
            print(f"Error deleting blob {blob_name}: {e}")
            return None

    print("All batch translation files have been deleted.")

Delete all the deleted measures

In [ ]:
def remove_deleted_measures_after_translation(df_translations,blob_client,translations_subset_blob_name,translations_container_client, timeformat):
    """
    Removes measures from the translations DataFrame that are not present in the measures from API DataFrame 
    retrieved from raw container.
    Args:
        df_translations (pd.DataFrame): DataFrame containing translated measures with a 'measureId' column.
        blob_client (BlobClient): The Azure Blob Storage client used to interact with the blob.
    Returns:
        None
    This function performs the following steps:
        1. Connects to the Azure Blob Storage container specified by `target_container_name`.
        2. Retrieves the 'Measures_from_API.parquet' file from the container.
        3. Loads the measures data from the parquet file into a DataFrame.
        4. Identifies and counts the measureIds in `df_translations` that are not present in the measures DataFrame.
        5. Filters out the rows in `df_translations` that have measureIds not present in the measures DataFrame.
        6. If any rows were filtered out, overwrites the existing 'Measures.parquet' file in the container with the filtered DataFrame.
        7. Prints the count of measureIds not present in the measures DataFrame and a message indicating whether any rows were deleted.
    """
    target_container_name = 'raw'
    blob_service_client = BlobServiceClient(
    account_url=f"https://<storage_account_name>.blob.core.windows.net", credential=credential)

    # Measures file
    source_container_client = blob_service_client.get_container_client(
        target_container_name)
    file_path = f"{folder_name}/Measures/Measures_from_API.parquet"
    json_blobs = source_container_client.list_blobs(name_starts_with=file_path)

    measures_blob = [blob for blob in json_blobs if blob.name.endswith("Measures_from_API.parquet")]
    if not measures_blob:
        print("No Measures_from_API.parquet file found in the source container.")
    blob = measures_blob[0]

    source_blob_client = source_container_client.get_blob_client(blob.name)
    blob_data = source_blob_client.download_blob().readall()
    df_measures = pd.read_parquet(BytesIO(blob_data))
    
    # Translations file
    # Measure Ids in translations file that are not present in Measures from API file (deleted measures) --> to obtain a count of deleted measures
    not_in_orig = ~df_translations['measureId'].isin(df_measures['measureId']) 
    not_in_orig_count = not_in_orig.sum()
    print(f"Count of measureIds in translated_measures not present in measures from API: {not_in_orig_count}. These are the measures that are deleted from the source.")

    # Save deleted measures subset to a folder
    deleted_measures = df_translations[~df_translations['measureId'].isin(df_measures['measureId'])]

    save_deleted_measures(deleted_measures,translations_container_client, translations_subset_blob_name, timeformat)

    filtered_df_translations = df_translations[df_translations['measureId'].isin(df_measures['measureId'])]
    if not_in_orig_count > 0:
        output_buffer = BytesIO()
        filtered_df_translations.to_parquet(output_buffer, index=False)
        output_buffer.seek(0)

        # Overwrite the existing Measures.parquet file in the translated-measures container
        blob_client.upload_blob(output_buffer, overwrite=True)
        print("The measures that are not present in the API data have been successfully removed from the Measures.parquet file in the translated-measures container.")
    else:
        print("No rows to delete; all measureIds from the API data are present in the translated Measures.parquet file.")

Merge all the translated batch files

In [ ]:
def merge_translated_measures_parquet_files(translations_container_name,translations_subset_blob_name,output_data_path_info):
    """ 
    Merges all the translated files as the translation occurs in batches.
    Merges the concatenated translation files with the previously translated file.
    If duplicates exists consider latest timestamp.

    Args:
        translations_container_name(str): Container to store the translated measures.
        output_data_path_info (str): The info file, which contains the path of saved the Azure ML data.

    Returns:
        None
    """

    file_name = "Measures.parquet"
    utc_time = datetime.now(timezone('UTC'))
    timeformat = utc_time.astimezone(timezone('Europe/Berlin')).strftime("%Y-%m-%d_%H:%M:%S")

    # Initialize the Blob Service Client
    blob_service_client = BlobServiceClient(
        account_url="https://<storage_account_name>.blob.core.windows.net", credential=credential)
    container_client = blob_service_client.get_container_client(
        container=translations_container_name)
    blob_client = container_client.get_blob_client(blob=file_name)

    try:
        parquet_blobs = [blob.name for blob in container_client.list_blobs(
            name_starts_with="translated/") if blob.name.endswith('.parquet')]

        # Loop through each parquet blob, download and process
        dataframes = []
        for blob_name in parquet_blobs:
            try:
                downloaded_blob = container_client.download_blob(blob_name)
                parquet_data = BytesIO(downloaded_blob.readall())
                df = pd.read_parquet(parquet_data)
                dataframes.append(df)
            except Exception as e:
                print(f"Error downloading or processing blob {blob_name}: {e}")

        # Condition to check if the dataframes list is empty
        # if not dataframes:
        #     print("No dataframes to merge. Exiting function. ")
        #     update_datastore_path(file_name, output_data_path_info)
        #     return

        combined_df = pd.concat(dataframes, ignore_index=True)

        copy_new_or_updated_translation_files(container_client, translations_subset_blob_name, combined_df, timeformat)

        if blob_client.exists():
            blob_data = blob_client.download_blob()
            df_existing = pd.read_parquet(BytesIO(blob_data.readall()))

            # Concatenate new dataframe to the existing translated measures parquet file
            all_data = pd.concat([combined_df, df_existing], ignore_index=True)
        else:
            all_data = combined_df

        # Converting date columns to datetime datatype
        all_data['lastUpdatedAt'] = pd.to_datetime(
            all_data['lastUpdatedAt'], errors='coerce')
        all_data['createdAt'] = all_data['createdAt'].apply(
            lambda x: x[:19] + 'Z' if isinstance(x, str) else x)
        all_data['createdAt'] = pd.to_datetime(all_data['createdAt'])
        all_data['endDate'] = pd.to_datetime(
            all_data['endDate'], errors='coerce')
        all_data['startDate'] = pd.to_datetime(
            all_data['startDate'], errors='coerce')

        all_data['projectTypes'] = all_data['projectTypes'].astype(str)
        all_data['status'] = all_data['status'].astype(str)
        all_data['hardnessGrade'] = all_data['hardnessGrade'].astype(str)
        all_data['priority'] = all_data['priority'].astype(str)
        all_data['simpleScore'] = all_data['simpleScore'].astype(float)

        all_data['latestTimestamp'] = all_data[[
            'createdAt', 'lastUpdatedAt']].max(axis=1)

        # Sort by measureId and latest timestamp, and keep the last occurrence of each measureId
        all_data = all_data.sort_values(
            by=['measureId', 'latestTimestamp'], ascending=[True, False])
        all_data = all_data.drop_duplicates(
            subset='measureId', keep='first').drop(columns='latestTimestamp')

        # Save the updated DataFrame to a Parquet file and upload it, overwriting the existing file if needed
        with BytesIO() as output_stream:
            all_data.to_parquet(output_stream, index=False, engine='pyarrow')
            output_stream.seek(0)
            blob_client.upload_blob(output_stream, overwrite=True)

        print("Translation files are merged to the original translations file.")

        # Delete the batch translation files after merging them to free up storage space
        delete_batch_translation_files(
            blob_service_client, translations_container_name)
        
        remove_deleted_measures_after_translation(all_data,blob_client,translations_subset_blob_name,container_client, timeformat)

        # Save adjusted file to translatedmeasures datastore
        update_datastore_path(file_name, output_data_path_info)

    except (Exception, ResourceNotFoundError) as e:
        print(f"An error occurred: {e}")